In [ ]:
import pandas as pd
from IPython.display import display

# Change file name if needed
csv_file = "predictions.csv"

df = pd.read_csv(csv_file)

display(f"File: {csv_file}")
display(f"Number of rows: {df.shape[0]}")
display(f"Number of columns: {df.shape[1]}")

display("Columns:")
display(pd.DataFrame({"Columns": df.columns}))

display("Head:")
display(df.head())

In [ ]:
# --- Split dataset ---
df_first32 = df.iloc[:32].copy()
df_remaining = df.iloc[32:].copy()

# Reset index for clean ID assignment
df_first32 = df_first32.reset_index(drop=True)
df_remaining = df_remaining.reset_index(drop=True)

# Add ID columns
df_first32["ID"] = df_first32.index + 1
df_remaining["ID"] = df_remaining.index + 1

# Move ID to first column
df_first32 = df_first32[["ID"] + [c for c in df_first32.columns if c != "ID"]]
df_remaining = df_remaining[["ID"] + [c for c in df_remaining.columns if c != "ID"]]

display("First 32 rows:")
display(df_first32.head())
display(f"Total in first group: {df_first32.shape[0]}")

display("Remaining rows:")
display(df_remaining.head())
display(f"Total in remaining group: {df_remaining.shape[0]}")

In [ ]:
import numpy as np

# --- Select relevant columns ---
cols = [
    "Ensemble_Average_PSA_Chloroform_ANI",
    "Ensemble_Average_Num_IMHB_Chloroform_ANI",
    "Ensemble_Average_RadiusOfGyration_Chloroform_ANI",
    "Predicted_P_appLog"
]

df_first_table = df_first32[["ID"] + cols].copy()
df_remaining_table = df_remaining[["ID"] + cols].copy()

# --- Reverse log10 transform ---
df_first_table["Predicted Papp (nm/s)"] = 10 ** df_first_table["Predicted_P_appLog"]
df_remaining_table["Predicted Papp (nm/s)"] = 10 ** df_remaining_table["Predicted_P_appLog"]

# --- Drop log column ---
df_first_table = df_first_table.drop(columns=["Predicted_P_appLog"])
df_remaining_table = df_remaining_table.drop(columns=["Predicted_P_appLog"])

# --- Rename columns ---
rename_dict = {
    "ID": "compound",
    "Ensemble_Average_PSA_Chloroform_ANI": "3D-PSA (Å²)",
    "Ensemble_Average_Num_IMHB_Chloroform_ANI": "IMHB (count)",
    "Ensemble_Average_RadiusOfGyration_Chloroform_ANI": "Rgyr (Å)"
}

df_first_table = df_first_table.rename(columns=rename_dict)
df_remaining_table = df_remaining_table.rename(columns=rename_dict)

# --- Reorder columns ---
ordered_cols = ["compound", "Predicted Papp (nm/s)", "3D-PSA (Å²)", "IMHB (count)", "Rgyr (Å)"]

df_first_table = df_first_table[ordered_cols]
df_remaining_table = df_remaining_table[ordered_cols]

# --- Round for clarity ---
round_dict = {
    "Predicted Papp (nm/s)": 2,
    "3D-PSA (Å²)": 2,
    "IMHB (count)": 2,
    "Rgyr (Å)": 2
}

df_first_table = df_first_table.round(round_dict)
df_remaining_table = df_remaining_table.round(round_dict)

# --- Create sorted versions ---
df_first_sorted = df_first_table.sort_values(by="Predicted Papp (nm/s)", ascending=False)
df_remaining_sorted = df_remaining_table.sort_values(by="Predicted Papp (nm/s)", ascending=False)

# --- Save all four files ---
df_first_table.to_csv("Table1_first32_original.csv", index=False)
df_first_sorted.to_csv("Table1_first32_sorted.csv", index=False)

df_remaining_table.to_csv("Table1_remaining_original.csv", index=False)
df_remaining_sorted.to_csv("Table1_remaining_sorted.csv", index=False)

display("Saved:")
display("  - Table1_first32_original.csv")
display("  - Table1_first32_sorted.csv")
display("  - Table1_remaining_original.csv")
display("  - Table1_remaining_sorted.csv")

display("Preview (First 32 Original):")
display(df_first_table.head())

display("Preview (First 32 Sorted):")
display(df_first_sorted.head())